# Brazil's top coffee municipalities → cadastral fazendas → coffee farms on OpenStreetMap

This notebook:

1. **Ranks Brazilian municipalities by coffee production** using IBGE's *Produção Agrícola Municipal* (PAM, SIDRA table 1613, "Café (em grão)", total / arábica / canéfora).
2. Maps the national picture and picks the **#1 municipality** (production averaged over the last few years, because coffee has a strong on/off biennial cycle).
3. **Breaks that municipality into cadastral rural properties (fazendas)** using the federal rural environmental registry, **CAR / SICAR** (public WFS at `geoserver.car.gov.br`), with an option to load a local CAR or INCRA **SIGEF** shapefile instead.
4. Pulls **OpenStreetMap** data through the Overpass API: land tagged as coffee (`crop=coffee`, `produce=coffee`, …) and named farms (`place=farm`, "Fazenda …").
5. Overlays the two to flag **coffee fazendas**: cadastral properties that contain OSM-mapped coffee or an OSM farm tagged as producing coffee.
6. Draws static maps on an OSM basemap and an interactive **Folium** map, and exports everything (CSV, GeoPackage, HTML, PNG).

### Data sources

| Layer | Source | Access |
|---|---|---|
| Coffee production by municipality | IBGE PAM, table 1613 | `servicodados.ibge.gov.br/api/v3/agregados` |
| Municipal and state boundaries | IBGE malhas API v3 | `servicodados.ibge.gov.br/api/v3/malhas` |
| Cadastral rural properties | SICAR (CAR), layer `sicar:sicar_imoveis_<uf>` | `geoserver.car.gov.br/geoserver/sicar/wfs` |
| Coffee land and farms | OpenStreetMap contributors (ODbL) | Overpass API |
| Basemaps | OpenStreetMap tiles; Esri imagery in the interactive map | tile servers |

### Caveats to read before trusting the output

* **CAR is self-declared.** Polygons can overlap, be pending review, or be imprecise. Cancelled records are removed. Totals that could double count use the *union* of geometries.
* **OSM coverage of coffee is partial.** Many plantations are mapped only as `landuse=farmland` or not at all. The notebook reports how much of IBGE's harvested coffee area OSM actually covers, so you can judge completeness. For wall-to-wall coverage, swap in a remote-sensing coffee mask (e.g. MapBiomas coffee class) in section 6.
* All public endpoints are rate-limited. Responses are cached under `cache/`, so re-runs are fast and polite.

## 0. Setup

In [ ]:
# If needed:  pip install geopandas shapely pyproj folium matplotlib contextily mapclassify requests pandas
import hashlib
import json
import re
import time
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import numpy as np
import pandas as pd
import requests
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from shapely.geometry import LineString, Point, Polygon, shape
from shapely.ops import polygonize, unary_union

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

In [ ]:
# ---------------- configuration ----------------
TOP_N = 15                 # municipalities to show in the ranking
AVG_YEARS = 3              # years averaged for the ranking (smooths the biennial cycle)
HISTORY_YEARS = 10         # years shown in the trend chart
MUNICIPALITY_OVERRIDE = None   # e.g. 3148103 to force a municipality (IBGE 7-digit code)

# Cadastral source: "sicar_wfs" (download live) or "local" (a file you downloaded yourself)
CADASTRE_SOURCE = "sicar_wfs"
CADASTRE_LOCAL_PATH = None     # e.g. "data/SHAPE_3148103/AREA_IMOVEL/AREA_IMOVEL.shp" or an INCRA SIGEF .zip
MIN_SHARE_INSIDE = 0.5         # a property belongs to the municipality if >= this share of its area is inside

# What counts as a coffee fazenda
MIN_COFFEE_HA = 0.5            # minimum OSM-mapped coffee area inside a property

CACHE_DIR = Path("cache");   CACHE_DIR.mkdir(exist_ok=True)
OUT_DIR = Path("outputs");   OUT_DIR.mkdir(exist_ok=True)

# Colors: validated categorical slots + one-hue sequential ramp (colorblind-checked)
C_ARABICA, C_CANEPHORA, C_OSM_COFFEE, C_FARM = "#2a78d6", "#eb6834", "#1baf7a", "#4a3aa7"
BLUE_RAMP = ["#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#184f95", "#0d366b"]
INK, INK_2, MUTED, GRID = "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 10,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK_2, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold", "axes.titlesize": 12,
})

In [ ]:
# ---------------- polite, cached HTTP ----------------
SESSION = requests.Session()
SESSION.headers["User-Agent"] = "brazil-coffee-fazendas-notebook/1.0 (research; python-requests)"


def fetch(url, params=None, data=None, timeout=300, retries=4, use_cache=True):
    """GET (or POST when `data` is given) with on-disk cache and exponential backoff."""
    key = hashlib.sha1(json.dumps([url, params, data], sort_keys=True, default=str).encode()).hexdigest()
    path = CACHE_DIR / key
    if use_cache and path.exists():
        return path.read_bytes()
    last_err = None
    for attempt in range(retries):
        try:
            if data is None:
                r = SESSION.get(url, params=params, timeout=timeout)
            else:
                r = SESSION.post(url, params=params, data=data, timeout=timeout)
            if r.status_code in (429, 502, 503, 504):
                raise requests.HTTPError(f"{r.status_code} from {url}", response=r)
            r.raise_for_status()
            path.write_bytes(r.content)
            return r.content
        except (requests.RequestException, OSError) as err:
            last_err = err
            wait = 2 ** (attempt + 1)
            print(f"  retry {attempt + 1}/{retries} in {wait}s: {err}")
            time.sleep(wait)
    raise RuntimeError(f"Failed to fetch {url}: {last_err}")


def fetch_json(url, **kw):
    return json.loads(fetch(url, **kw))

## 1. Coffee production by municipality (IBGE PAM)

Variable and category codes are discovered from the table's metadata by name, so the notebook keeps working if IBGE renumbers them. PAM reports production in **tonnes of green (processed) coffee**; we also convert to the trade unit of **60 kg bags**.

In [ ]:
IBGE = "https://servicodados.ibge.gov.br/api/v3"
PAM_TABLE = 1613

meta = fetch_json(f"{IBGE}/agregados/{PAM_TABLE}/metadados")
variables = {v["id"]: v["nome"] for v in meta["variaveis"]}


def find_id(items, pattern, fallback):
    for k, name in items.items():
        if re.search(pattern, name, flags=re.I):
            return k
    print(f"  ! '{pattern}' not found in metadata, using fallback id {fallback}")
    return fallback


V_QTY = find_id(variables, r"^Quantidade produzida$", 214)
V_AREA = find_id(variables, r"^[ÁA]rea colhida$", 216)
V_VALUE = find_id(variables, r"^Valor da produ[çc][ãa]o$", 215)

prod_class = next(c for c in meta["classificacoes"] if c["id"] == 82)
categories = {c["id"]: c["nome"] for c in prod_class["categorias"]}
CAT_TOTAL = find_id(categories, r"Caf[ée].*Total", 2723)
CAT_ARABICA = find_id(categories, r"Caf[ée].*Ar[áa]bica", 31619)
CAT_CANEPHORA = find_id(categories, r"Caf[ée].*Canephora|Caf[ée].*Can[ée]fora|Caf[ée].*Robusta", 31620)

periods = sorted(int(p["id"]) for p in fetch_json(f"{IBGE}/agregados/{PAM_TABLE}/periodos"))
LAST_YEAR = periods[-1]
RANK_YEARS = periods[-AVG_YEARS:]
TREND_YEARS = periods[-HISTORY_YEARS:]

print("Variables :", {V_QTY: variables.get(V_QTY), V_AREA: variables.get(V_AREA), V_VALUE: variables.get(V_VALUE)})
print("Categories:", {c: categories.get(c) for c in (CAT_TOTAL, CAT_ARABICA, CAT_CANEPHORA)})
print(f"Latest PAM year: {LAST_YEAR}; ranking uses {RANK_YEARS}")

In [ ]:
def parse_value(v):
    """IBGE codes: '-' = zero, '...'/'..'/'X' = not available / suppressed."""
    if v is None:
        return np.nan
    v = str(v).strip()
    if v == "-":
        return 0.0
    try:
        return float(v)
    except ValueError:
        return np.nan


def pam_values(var_id, cat_id, years):
    """One request per year keeps each response well under the API's size limit."""
    rows = []
    for y in years:
        url = (f"{IBGE}/agregados/{PAM_TABLE}/periodos/{y}/variaveis/{var_id}"
               f"?localidades=N6[all]&classificacao=82[{cat_id}]")
        for var in fetch_json(url):
            for res in var["resultados"]:
                for s in res["series"]:
                    loc = s["localidade"]
                    for per, val in s["serie"].items():
                        rows.append((int(loc["id"]), loc["nome"], int(per), parse_value(val)))
    return pd.DataFrame(rows, columns=["code", "name", "year", "value"])


qty_total = pam_values(V_QTY, CAT_TOTAL, sorted(set(TREND_YEARS) | set(RANK_YEARS)))
qty_arab = pam_values(V_QTY, CAT_ARABICA, RANK_YEARS)
qty_cane = pam_values(V_QTY, CAT_CANEPHORA, RANK_YEARS)
area_total = pam_values(V_AREA, CAT_TOTAL, RANK_YEARS)
value_total = pam_values(V_VALUE, CAT_TOTAL, [LAST_YEAR])
print(f"{qty_total['code'].nunique():,} municipalities, {len(qty_total):,} municipality-years")

In [ ]:
def mean_over(df, years):
    return df[df["year"].isin(years)].groupby("code")["value"].mean()


names = qty_total.drop_duplicates("code").set_index("code")["name"]
muni = pd.DataFrame({
    "name_uf": names,
    "prod_t": mean_over(qty_total, RANK_YEARS),
    "arabica_t": mean_over(qty_arab, RANK_YEARS),
    "canephora_t": mean_over(qty_cane, RANK_YEARS),
    "harvested_ha": mean_over(area_total, RANK_YEARS),
    f"prod_t_{LAST_YEAR}": qty_total[qty_total["year"] == LAST_YEAR].set_index("code")["value"],
    f"value_kBRL_{LAST_YEAR}": value_total.set_index("code")["value"],
})
muni[["municipality", "uf"]] = muni["name_uf"].str.rsplit(" - ", n=1, expand=True)
muni["bags_60kg"] = muni["prod_t"] * 1000 / 60
muni["yield_bags_ha"] = muni["bags_60kg"] / muni["harvested_ha"]
muni["arabica_share"] = muni["arabica_t"] / (muni["arabica_t"] + muni["canephora_t"])
muni = muni[muni["prod_t"] > 0].sort_values("prod_t", ascending=False)
muni["rank"] = np.arange(1, len(muni) + 1)
muni["share_brazil_%"] = 100 * muni["prod_t"] / muni["prod_t"].sum()

top = muni.head(TOP_N)
cols = ["rank", "municipality", "uf", "prod_t", "bags_60kg", "share_brazil_%", "arabica_share",
        "harvested_ha", "yield_bags_ha", f"prod_t_{LAST_YEAR}"]
print(f"{len(muni):,} municipalities produced coffee (mean {RANK_YEARS[0]}–{RANK_YEARS[-1]}); "
      f"top {TOP_N} = {top['share_brazil_%'].sum():.1f}% of Brazil")
muni.to_csv(OUT_DIR / "coffee_municipality_ranking.csv", index_label="ibge_code")
top[cols].style.format({
    "prod_t": "{:,.0f}", "bags_60kg": "{:,.0f}", "share_brazil_%": "{:.2f}", "arabica_share": "{:.0%}",
    "harvested_ha": "{:,.0f}", "yield_bags_ha": "{:.1f}", f"prod_t_{LAST_YEAR}": "{:,.0f}",
}).hide(axis="index")

In [ ]:
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(14, 6.2), gridspec_kw={"width_ratios": [1.35, 1]})

# (a) ranking: stacked arábica / canéfora, in thousand bags
t = top.iloc[::-1]
labels = [f"{r}. {m} ({u})" for r, m, u in zip(t["rank"], t["municipality"], t["uf"])]
arab = t["arabica_t"].fillna(0) * 1000 / 60 / 1e3
cane = t["canephora_t"].fillna(0) * 1000 / 60 / 1e3
ax.barh(labels, arab, color=C_ARABICA, height=0.7, label="Arábica", edgecolor="white", linewidth=1)
ax.barh(labels, cane, left=arab, color=C_CANEPHORA, height=0.7, label="Canéfora (conilon / robusta)",
        edgecolor="white", linewidth=1)
for y, total in enumerate(arab + cane):
    ax.text(total, y, f"  {total:,.0f}", va="center", fontsize=8.5, color=INK_2)
ax.set_xlabel(f"thousand 60-kg bags per year (mean {RANK_YEARS[0]}–{RANK_YEARS[-1]})")
ax.set_title(f"Top {TOP_N} coffee-producing municipalities in Brazil", loc="left")
ax.xaxis.grid(True, color=GRID, lw=0.6); ax.set_axisbelow(True)
ax.tick_params(axis="y", colors=INK, length=0)
ax.legend(frameon=False, loc="lower right")
ax.margins(x=0.12)

# (b) biennial trend for the top 5
trend = qty_total[qty_total["code"].isin(top.index[:5])]
series_colors = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
for c, code_ in zip(series_colors, top.index[:5]):
    s = trend[trend["code"] == code_].sort_values("year")
    ax2.plot(s["year"], s["value"] * 1000 / 60 / 1e3, color=c, lw=2, marker="o", ms=4)
    ax2.annotate(muni.loc[code_, "municipality"], (s["year"].iloc[-1], s["value"].iloc[-1] * 1000 / 60 / 1e3),
                 xytext=(6, 0), textcoords="offset points", va="center", fontsize=8.5, color=INK_2)
ax2.set_title("Production of the top 5, by year", loc="left")
ax2.set_ylabel("thousand 60-kg bags")
ax2.yaxis.grid(True, color=GRID, lw=0.6); ax2.set_axisbelow(True)
ax2.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax2.margins(x=0.25)
ax2.legend([Line2D([], [], color=c, lw=2) for c in series_colors],
           [muni.loc[c, "municipality"] for c in top.index[:5]], frameon=False, fontsize=8.5, loc="upper left")
fig.text(0.01, -0.01, f"Source: IBGE – Produção Agrícola Municipal, table {PAM_TABLE}.", color=MUTED, fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / "01_top_municipalities.png", bbox_inches="tight")
plt.show()

### National map

In [ ]:
def ibge_mesh(path, quality="minima", **params):
    q = {"formato": "application/vnd.geo+json", "qualidade": quality, **params}
    gdf = gpd.GeoDataFrame.from_features(fetch_json(f"{IBGE}/malhas/{path}", params=q)["features"], crs=4674)
    gdf["code"] = gdf["codarea"].astype(int)
    return gdf


states = ibge_mesh("paises/BR", intrarregiao="UF")
munis = ibge_mesh("paises/BR", intrarregiao="municipio").merge(
    muni[["prod_t", "rank", "municipality", "uf"]], left_on="code", right_index=True, how="left")
munis["prod_t"] = munis["prod_t"].fillna(0)

BINS = [0, 500, 2_000, 10_000, 25_000, np.inf]
BIN_LABELS = ["< 500 t", "500 – 2,000 t", "2,000 – 10,000 t", "10,000 – 25,000 t", "≥ 25,000 t"]
munis["cls"] = pd.cut(munis["prod_t"], BINS, labels=False, right=False)
munis.loc[munis["prod_t"] <= 0, "cls"] = -1
ramp = BLUE_RAMP[1:]


def draw_national(ax, bounds=None, label_top=0):
    munis[munis["cls"] == -1].plot(ax=ax, color="#f0efec", linewidth=0)
    for k, c in enumerate(ramp):
        sub = munis[munis["cls"] == k]
        if len(sub):
            sub.plot(ax=ax, color=c, linewidth=0)
    states.boundary.plot(ax=ax, color="white", linewidth=0.9)
    states.boundary.plot(ax=ax, color=MUTED, linewidth=0.3)
    if label_top:
        for _, r in munis[munis["rank"] <= label_top].iterrows():
            p = r.geometry.representative_point()
            ax.annotate(f"{int(r['rank'])}", (p.x, p.y), fontsize=8, fontweight="bold", color=INK, ha="center",
                        va="center", path_effects=[pe.withStroke(linewidth=2.5, foreground="white")])
    if bounds is not None:
        ax.set_xlim(bounds[0], bounds[2]); ax.set_ylim(bounds[1], bounds[3])
    ax.set_axis_off()


fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 7.5), gridspec_kw={"width_ratios": [1, 1.25]})
draw_national(a1)
zoom = munis[munis["rank"] <= 30].total_bounds
pad = 1.0
zoom = [zoom[0] - pad, zoom[1] - pad, zoom[2] + pad, zoom[3] + pad]
a1.add_patch(plt.Rectangle((zoom[0], zoom[1]), zoom[2] - zoom[0], zoom[3] - zoom[1], fill=False, ec=INK, lw=1))
a1.set_title(f"Coffee production by municipality (mean {RANK_YEARS[0]}–{RANK_YEARS[-1]})", loc="left")
draw_national(a2, zoom, label_top=TOP_N)
for _, r in states.iterrows():
    p = r.geometry.representative_point()
    if zoom[0] < p.x < zoom[2] and zoom[1] < p.y < zoom[3]:
        a2.annotate(r.get("sigla", ""), (p.x, p.y), fontsize=11, color=MUTED, ha="center", alpha=0.8)
a2.set_title(f"Coffee belt – numbers are the top-{TOP_N} ranks", loc="left")
handles = [Patch(fc="#f0efec", label="no production")] + [Patch(fc=c, label=l) for c, l in zip(ramp, BIN_LABELS)]
a1.legend(handles=handles, loc="upper left", bbox_to_anchor=(0, 0), ncol=3, frameon=False,
           title="tonnes of coffee per year", fontsize=8.5, alignment="left")
fig.text(0.01, 0.01, "Source: IBGE PAM (table 1613) and IBGE municipal mesh.", color=MUTED, fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / "02_national_map.png", bbox_inches="tight")
plt.show()

## 2. The #1 municipality

The municipality with the highest mean production is selected (set `MUNICIPALITY_OVERRIDE` to study another one). Its boundary is fetched at full resolution and a local UTM projection (SIRGAS 2000) is used for all area calculations.

In [ ]:
TARGET = int(MUNICIPALITY_OVERRIDE or muni.index[0])
target = muni.loc[TARGET]
UF = target["uf"]
NAME = target["municipality"]

boundary = ibge_mesh(f"municipios/{TARGET}", quality="maxima")
boundary["name"] = NAME
UTM = boundary.estimate_utm_crs(datum_name="SIRGAS 2000")
boundary_utm = boundary.to_crs(UTM)
MUNI_HA = boundary_utm.area.iloc[0] / 1e4
BBOX = boundary.total_bounds          # lon/lat: minx, miny, maxx, maxy

print(f"#{int(target['rank'])}: {NAME} ({UF}), IBGE {TARGET}")
print(f"  mean production     : {target['prod_t']:,.0f} t  ≈ {target['bags_60kg']:,.0f} bags/yr "
      f"({target['share_brazil_%']:.2f}% of Brazil)")
print(f"  harvested coffee    : {target['harvested_ha']:,.0f} ha  ({target['yield_bags_ha']:.1f} bags/ha)")
print(f"  arábica share       : {target['arabica_share']:.0%}")
print(f"  municipality area   : {MUNI_HA:,.0f} ha; coffee ≈ {100 * target['harvested_ha'] / MUNI_HA:.1f}% of it")
print(f"  projected CRS       : {UTM}")

## 3. Cadastral fazendas (CAR / SICAR)

Rural properties are downloaded from the SICAR WFS for the municipality's bounding box, page by page, and then kept if at least `MIN_SHARE_INSIDE` of their area falls inside the municipality. The full (unclipped) property polygons are kept for display.

If the SICAR server is slow or down, download the municipality's shapefile from the [CAR public portal](https://consultapublica.car.gov.br/publico/imoveis/index) (or the state's INCRA SIGEF file from [certificacao.incra.gov.br](https://certificacao.incra.gov.br/csv_shp/export_shp.py)), then set `CADASTRE_SOURCE = "local"` and `CADASTRE_LOCAL_PATH`.

In [ ]:
SICAR_WFS = "https://geoserver.car.gov.br/geoserver/sicar/wfs"


def fetch_wfs_bbox(url, type_name, bbox, page=2000, max_pages=200):
    """WFS 2.0 paging. 'EPSG:4326' (not the URN form) keeps GeoServer in lon/lat axis order."""
    frames = []
    for i in range(max_pages):
        params = {
            "service": "WFS", "version": "2.0.0", "request": "GetFeature", "typeNames": type_name,
            "outputFormat": "application/json", "srsName": "EPSG:4326",
            "bbox": ",".join(f"{v:.6f}" for v in bbox) + ",EPSG:4326",
            "count": page, "startIndex": i * page,
        }
        js = fetch_json(url, params=params, timeout=600)
        feats = js.get("features", [])
        print(f"  page {i + 1}: {len(feats)} features")
        if feats:
            frames.append(gpd.GeoDataFrame.from_features(feats, crs=4326))
        if len(feats) < page:
            break
    if not frames:
        return gpd.GeoDataFrame(geometry=[], crs=4326)
    return pd.concat(frames, ignore_index=True)


def fix_axis_order(gdf, ref_bounds):
    """Swap x/y if a server answered in lat/lon order despite the request."""
    if gdf.empty:
        return gdf
    minx, miny, maxx, maxy = gdf.total_bounds
    rminx, rminy, rmaxx, rmaxy = ref_bounds
    overlaps = not (maxx < rminx or minx > rmaxx or maxy < rminy or miny > rmaxy)
    if not overlaps and (rminy - 1 < minx < rmaxy + 1):
        print("  ! swapping axis order")
        from shapely.ops import transform
        gdf = gdf.set_geometry(gdf.geometry.map(lambda g: transform(lambda x, y, z=None: (y, x), g)))
    return gdf


if CADASTRE_SOURCE == "local":
    raw = gpd.read_file(CADASTRE_LOCAL_PATH, bbox=tuple(BBOX))
else:
    print(f"Downloading CAR properties for {NAME} ({UF}) …")
    raw = fetch_wfs_bbox(SICAR_WFS, f"sicar:sicar_imoveis_{UF.lower()}", BBOX)
    raw = fix_axis_order(raw, BBOX)
raw = raw.set_crs(4326, allow_override=False) if raw.crs is None else raw
print(f"{len(raw):,} records in the bounding box; columns: {list(raw.columns)}")

In [ ]:
def pick(df, *candidates):
    """First existing column (case-insensitive) among candidates, else None."""
    lower = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower:
            return df[lower[c.lower()]]
    return pd.Series([None] * len(df), index=df.index)


cad = gpd.GeoDataFrame({
    "property_id": pick(raw, "cod_imovel", "COD_IMOVEL", "codigo_imo", "parcela_co", "codigo"),
    "status": pick(raw, "status_imovel", "des_condic", "situacao", "SITUACAO", "status"),
    "kind": pick(raw, "tipo_imovel", "TIPO_IMOVE", "ind_tipo"),
    "declared_ha": pd.to_numeric(pick(raw, "area", "num_area", "NUM_AREA", "area_ha"), errors="coerce"),
    "fiscal_modules": pd.to_numeric(pick(raw, "m_fiscal", "mod_fiscal", "num_modulo", "NUM_MODULO"), errors="coerce"),
    "declared_name": pick(raw, "nome_imovel", "nom_imovel", "nome_area", "denominacao"),
}, geometry=raw.geometry.make_valid(), crs=raw.crs)
cad = cad[cad.geometry.notna() & ~cad.geometry.is_empty]
cad = cad[cad.geom_type.isin(["Polygon", "MultiPolygon", "GeometryCollection"])]
cad["geometry"] = cad.geometry.map(
    lambda g: unary_union([p for p in getattr(g, "geoms", [g]) if p.geom_type in ("Polygon", "MultiPolygon")])
    if g.geom_type == "GeometryCollection" else g)
cad = cad[~cad.geometry.is_empty]

# drop cancelled registrations
cancelled = cad["status"].astype(str).str.upper().str.match(r"^(CA|CANCEL)")
print(f"dropping {cancelled.sum():,} cancelled records")
cad = cad[~cancelled]

# duplicates: the WFS pages or overlapping source rows can repeat an id
if cad["property_id"].notna().any():
    cad = cad.drop_duplicates("property_id")
cad["property_id"] = cad["property_id"].fillna(pd.Series([f"P{i:05d}" for i in range(len(cad))], index=cad.index))

# keep properties mostly inside the municipality
cad_utm = cad.to_crs(UTM)
cad_utm["area_ha"] = cad_utm.area / 1e4
inside = cad_utm.geometry.intersection(boundary_utm.geometry.iloc[0]).area / 1e4
cad_utm["share_inside"] = (inside / cad_utm["area_ha"]).clip(0, 1)
cad_utm = cad_utm[cad_utm["share_inside"] >= MIN_SHARE_INSIDE].copy()

# size classes in módulos fiscais (Brazil's legal size classes); fall back to hectares
mf = (cad_utm["declared_ha"] / cad_utm["fiscal_modules"]).replace([np.inf, -np.inf], np.nan)
MF_HA = mf[(mf > 1) & (mf < 150)].median()
if pd.notna(MF_HA):
    cad_utm["mf"] = cad_utm["area_ha"] / MF_HA
    SIZE_BINS, SIZE_LABELS = [0, 1, 4, 15, np.inf], ["Minifúndio (< 1 MF)", "Small (1–4 MF)",
                                                     "Medium (4–15 MF)", "Large (> 15 MF)"]
    size_basis = cad_utm["mf"]
    print(f"módulo fiscal ≈ {MF_HA:.0f} ha")
else:
    SIZE_BINS, SIZE_LABELS = [0, 20, 100, 500, np.inf], ["< 20 ha", "20–100 ha", "100–500 ha", "> 500 ha"]
    size_basis = cad_utm["area_ha"]
cad_utm["size_class"] = pd.cut(size_basis, SIZE_BINS, labels=SIZE_LABELS, right=False)

covered_ha = unary_union(cad_utm.geometry.intersection(boundary_utm.geometry.iloc[0]).values).area / 1e4
print(f"{len(cad_utm):,} cadastral properties in {NAME}; they cover {covered_ha:,.0f} ha "
      f"= {100 * covered_ha / MUNI_HA:.0f}% of the municipality (union, overlaps counted once)")
cad_utm.groupby("size_class", observed=False).agg(
    properties=("area_ha", "size"), total_ha=("area_ha", "sum"), median_ha=("area_ha", "median")
).style.format({"total_ha": "{:,.0f}", "median_ha": "{:,.1f}"})

## 4. OpenStreetMap: coffee land and farms

Two Overpass queries over the municipality's bounding box:

* **coffee** — anything tagged `crop`, `produce` or `trees` containing *coffee/café/coffea*, plus farmland/orchards whose name mentions café;
* **farms** — `place=farm`, `landuse=farmyard`, and anything named *Fazenda / Sítio / Chácara / Estância / Granja*.

Ways and multipolygon relations are rebuilt into polygons; nodes become points.

In [ ]:
OVERPASS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]
S, W, N, E = BBOX[1], BBOX[0], BBOX[3], BBOX[2]
bb = f"({S:.6f},{W:.6f},{N:.6f},{E:.6f})"

Q_COFFEE = f"""[out:json][timeout:600];
(
  nwr["crop"~"coffee|caf[eé]",i]{bb};
  nwr["produce"~"coffee|caf[eé]",i]{bb};
  nwr["trees"~"coffea|coffee",i]{bb};
  nwr["landuse"~"^(farmland|orchard|plantation)$"]["name"~"caf[eé]",i]{bb};
);
out tags geom;"""

Q_FARMS = f"""[out:json][timeout:600];
(
  nwr["place"="farm"]{bb};
  nwr["landuse"="farmyard"]{bb};
  nwr["name"~"^(Fazenda|Faz\\\\.|S[ií]tio|Ch[aá]cara|Est[aâ]ncia|Granja) ",i]{bb};
);
out tags geom;"""


def overpass(query):
    last = None
    for url in OVERPASS:
        try:
            return fetch_json(url, data={"data": query}, timeout=900)
        except RuntimeError as err:
            last = err
            print(f"  {url} failed, trying next mirror")
    raise last


def element_geometry(el):
    if el["type"] == "node":
        return Point(el["lon"], el["lat"])
    if el["type"] == "way" and el.get("geometry"):
        pts = [(p["lon"], p["lat"]) for p in el["geometry"] if p]
        if len(pts) >= 4 and pts[0] == pts[-1]:
            return Polygon(pts)
        return LineString(pts) if len(pts) >= 2 else None
    if el["type"] == "relation":
        outer, inner = [], []
        for m in el.get("members", []):
            if m.get("type") == "way" and m.get("geometry"):
                line = LineString([(p["lon"], p["lat"]) for p in m["geometry"] if p])
                (inner if m.get("role") == "inner" else outer).append(line)
        polys = list(polygonize(unary_union(outer))) if outer else []
        if not polys:
            return None
        geom = unary_union(polys)
        if inner:
            geom = geom.difference(unary_union(list(polygonize(unary_union(inner)))))
        return geom
    return None


def overpass_gdf(query):
    js = overpass(query)
    rows = []
    for el in js.get("elements", []):
        g = element_geometry(el)
        if g is None or g.is_empty:
            continue
        tags = el.get("tags", {})
        rows.append({"osm_id": f"{el['type']}/{el['id']}", "name": tags.get("name"),
                     "landuse": tags.get("landuse"), "crop": tags.get("crop"), "produce": tags.get("produce"),
                     "place": tags.get("place"), "tags": json.dumps(tags, ensure_ascii=False), "geometry": g})
    gdf = gpd.GeoDataFrame(rows, geometry="geometry", crs=4326) if rows else \
        gpd.GeoDataFrame(columns=["osm_id", "name", "tags", "geometry"], geometry="geometry", crs=4326)
    return gdf.to_crs(UTM)


muni_geom = boundary_utm.geometry.iloc[0]
osm_coffee = overpass_gdf(Q_COFFEE)
osm_coffee = osm_coffee[osm_coffee.intersects(muni_geom)]
coffee_poly = osm_coffee[osm_coffee.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
coffee_poly["geometry"] = coffee_poly.intersection(muni_geom)
coffee_pts = osm_coffee[~osm_coffee.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
coffee_pts["geometry"] = coffee_pts.geometry.representative_point()

osm_farms = overpass_gdf(Q_FARMS)
osm_farms = osm_farms[osm_farms.intersects(muni_geom)].copy()
osm_farms["point"] = osm_farms.geometry.representative_point()

coffee_union = unary_union(coffee_poly.geometry.values) if len(coffee_poly) else None
OSM_COFFEE_HA = coffee_union.area / 1e4 if coffee_union is not None else 0.0
print(f"OSM coffee polygons: {len(coffee_poly):,}  ({OSM_COFFEE_HA:,.0f} ha, overlaps counted once)")
print(f"OSM coffee points  : {len(coffee_pts):,}")
print(f"OSM farms          : {len(osm_farms):,}  ({osm_farms['name'].notna().sum():,} named)")
print(f"OSM covers {100 * OSM_COFFEE_HA / target['harvested_ha']:.0f}% of IBGE's harvested coffee area "
      f"({target['harvested_ha']:,.0f} ha)")

## 5. Which fazendas grow coffee?

For each cadastral property we compute the OSM-mapped coffee area inside it and the names of OSM farms located in it. A property is a **coffee fazenda** if it holds at least `MIN_COFFEE_HA` of mapped coffee, or contains an OSM feature tagged as producing coffee.

In [ ]:
props = cad_utm[["property_id", "status", "kind", "declared_name", "area_ha", "size_class", "geometry"]].copy()

# coffee area per property
if len(coffee_poly):
    dissolved = gpd.GeoDataFrame(geometry=[coffee_union], crs=UTM).explode(index_parts=False)
    ov = gpd.overlay(props[["property_id", "geometry"]], dissolved, how="intersection", keep_geom_type=True)
    ov["coffee_ha"] = ov.area / 1e4
    props = props.merge(ov.groupby("property_id")["coffee_ha"].sum(), on="property_id", how="left")
props["coffee_ha"] = props.get("coffee_ha", pd.Series(0.0, index=props.index)).fillna(0.0)
props["coffee_share"] = (props["coffee_ha"] / props["area_ha"]).clip(0, 1)

# OSM coffee points (e.g. a farm node tagged produce=coffee) inside properties
if len(coffee_pts):
    hit = gpd.sjoin(coffee_pts[["osm_id", "geometry"]], props[["property_id", "geometry"]], predicate="within")
    props["osm_coffee_tag"] = props["property_id"].isin(hit["property_id"])
else:
    props["osm_coffee_tag"] = False

# OSM farm names inside properties
fp = gpd.GeoDataFrame(osm_farms[["osm_id", "name"]], geometry=osm_farms["point"], crs=UTM)
fp = fp[fp["name"].notna()]
if len(fp):
    j = gpd.sjoin(fp, props[["property_id", "geometry"]], predicate="within")
    names_by = j.groupby("property_id")["name"].agg(lambda s: "; ".join(sorted(set(s))))
    props = props.merge(names_by.rename("osm_farm_names"), on="property_id", how="left")
else:
    props["osm_farm_names"] = None

props["is_coffee"] = (props["coffee_ha"] >= MIN_COFFEE_HA) | props["osm_coffee_tag"]
props["label"] = props["osm_farm_names"].fillna(props["declared_name"]).fillna(props["property_id"])
props = gpd.GeoDataFrame(props, geometry="geometry", crs=UTM)

coffee_farms = props[props["is_coffee"]].sort_values("coffee_ha", ascending=False).reset_index(drop=True)
coffee_farms["coffee_rank"] = np.arange(1, len(coffee_farms) + 1)

print(f"{len(coffee_farms):,} of {len(props):,} cadastral properties are coffee fazendas "
      f"({coffee_farms['area_ha'].sum():,.0f} ha of property, {coffee_farms['coffee_ha'].sum():,.0f} ha of mapped coffee)")
print(f"{props['osm_farm_names'].notna().sum():,} properties matched to a named OSM farm")
coffee_farms.head(20)[["coffee_rank", "label", "property_id", "size_class", "area_ha", "coffee_ha", "coffee_share"]] \
    .style.format({"area_ha": "{:,.1f}", "coffee_ha": "{:,.1f}", "coffee_share": "{:.0%}"}).hide(axis="index")

In [ ]:
# Coffee by property size class — who grows the mapped coffee?
by_size = props.groupby("size_class", observed=False).agg(
    properties=("property_id", "size"), coffee_fazendas=("is_coffee", "sum"),
    property_ha=("area_ha", "sum"), coffee_ha=("coffee_ha", "sum"))
by_size["% of properties growing coffee"] = 100 * by_size["coffee_fazendas"] / by_size["properties"]
by_size["% of mapped coffee"] = 100 * by_size["coffee_ha"] / max(by_size["coffee_ha"].sum(), 1e-9)
by_size.style.format({"property_ha": "{:,.0f}", "coffee_ha": "{:,.0f}",
                      "% of properties growing coffee": "{:.0f}", "% of mapped coffee": "{:.0f}"})

## 6. Maps

Static maps sit on an OpenStreetMap basemap (via `contextily`, drawn in Web Mercator). If the tile server can't be reached the maps are still drawn, without a basemap.

> **Swapping in a remote-sensing coffee mask:** replace `coffee_poly` in section 4 with polygons vectorised from a coffee classification raster (e.g. MapBiomas' coffee class clipped to the municipality), then re-run sections 5–7.

In [ ]:
try:
    import contextily as ctx
except ImportError:
    ctx = None
WEB = 3857
b_web = boundary.to_crs(WEB)
props_web = props.to_crs(WEB)
coffee_web = coffee_farms.to_crs(WEB)
coffee_poly_web = coffee_poly.to_crs(WEB)
farms_web = gpd.GeoDataFrame(osm_farms.drop(columns="geometry"), geometry="point", crs=UTM).to_crs(WEB)


def basemap(ax, alpha=0.55, provider=None):
    if ctx is None:
        return
    try:
        ctx.add_basemap(ax, crs=WEB, source=provider or ctx.providers.OpenStreetMap.Mapnik, alpha=alpha,
                        attribution="© OpenStreetMap contributors", attribution_size=7)
    except Exception as err:            # offline / tile server refused
        print(f"  (basemap skipped: {type(err).__name__})")


def frame(ax, gdf, pad=0.03):
    x0, y0, x1, y1 = gdf.total_bounds
    dx, dy = (x1 - x0) * pad, (y1 - y0) * pad
    ax.set_xlim(x0 - dx, x1 + dx); ax.set_ylim(y0 - dy, y1 + dy); ax.set_axis_off()


def scalebar(ax, km=5):
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    # Web-Mercator metres are stretched by 1/cos(lat)
    lat = np.degrees(2 * np.arctan(np.exp(((y0 + y1) / 2) / 6378137)) - np.pi / 2)
    L = km * 1000 / np.cos(np.radians(lat))
    xs, ys = x0 + 0.04 * (x1 - x0), y0 + 0.04 * (y1 - y0)
    ax.plot([xs, xs + L], [ys, ys], color=INK, lw=3, solid_capstyle="butt",
            path_effects=[pe.withStroke(linewidth=5, foreground="white")])
    ax.text(xs + L / 2, ys + 0.012 * (y1 - y0), f"{km} km", ha="center", fontsize=8, color=INK,
            path_effects=[pe.withStroke(linewidth=2.5, foreground="white")])

In [ ]:
# Map A — the municipality broken into cadastral fazendas, coloured by size class
size_colors = dict(zip(SIZE_LABELS, BLUE_RAMP[1::1][:len(SIZE_LABELS)]))
fig, ax = plt.subplots(figsize=(11, 11))
for lab, col in size_colors.items():
    sub = props_web[props_web["size_class"] == lab]
    if len(sub):
        sub.plot(ax=ax, color=col, alpha=0.75, edgecolor="white", linewidth=0.35)
b_web.boundary.plot(ax=ax, color=INK, linewidth=1.6)
frame(ax, b_web)
basemap(ax, alpha=0.6)
scalebar(ax)
ax.set_title(f"{NAME} ({UF}) — {len(props):,} cadastral rural properties (CAR)", loc="left")
counts = props["size_class"].value_counts()
ax.legend(handles=[Patch(fc=c, alpha=0.75, label=f"{l}  ·  {counts.get(l, 0):,}") for l, c in size_colors.items()]
          + [Line2D([], [], color=INK, lw=1.6, label="Municipal boundary (IBGE)")],
          loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=9, title="Property size",
          alignment="left")
fig.text(0.02, 0.02, "Properties: SICAR/CAR (self-declared). Basemap © OpenStreetMap contributors.", color=MUTED, fontsize=8)
fig.savefig(OUT_DIR / "03_cadastral_fazendas.png", bbox_inches="tight")
plt.show()

In [ ]:
# Map B — coffee fazendas
SHARE_BINS = [0, 0.1, 0.25, 0.5, 1.01]
SHARE_LABELS = ["< 10% coffee", "10–25%", "25–50%", "≥ 50%"]
share_colors = BLUE_RAMP[2:6]
coffee_web["share_cls"] = pd.cut(coffee_web["coffee_share"], SHARE_BINS, labels=False, right=False)

fig, (ax, side) = plt.subplots(1, 2, figsize=(15, 11), gridspec_kw={"width_ratios": [2.6, 1]})
side.set_axis_off()
props_web[~props_web["is_coffee"]].plot(ax=ax, facecolor="none", edgecolor="#9d9b94", linewidth=0.3)
for k, c in enumerate(share_colors):
    sub = coffee_web[coffee_web["share_cls"] == k]
    if len(sub):
        sub.plot(ax=ax, color=c, alpha=0.8, edgecolor="white", linewidth=0.5)
if len(coffee_poly_web):
    coffee_poly_web.plot(ax=ax, facecolor="none", edgecolor=C_CANEPHORA, linewidth=0.6, hatch="////")
if len(farms_web):
    farms_web.plot(ax=ax, color=C_FARM, markersize=10, edgecolor="white", linewidth=0.4, zorder=5)
b_web.boundary.plot(ax=ax, color=INK, linewidth=1.6)
for _, r in coffee_web.head(10).iterrows():
    p = r.geometry.representative_point()
    ax.annotate(str(r["coffee_rank"]), (p.x, p.y), ha="center", va="center", fontsize=9, fontweight="bold",
                color=INK, path_effects=[pe.withStroke(linewidth=3, foreground="white")], zorder=6)
frame(ax, b_web)
basemap(ax, alpha=0.5)
scalebar(ax)
ax.set_title(f"{NAME} ({UF}) — {len(coffee_farms):,} coffee fazendas "
             f"(cadastral properties containing OSM-mapped coffee)", loc="left")
handles = [Patch(fc=c, alpha=0.8, label=l) for c, l in zip(share_colors, SHARE_LABELS)] + [
    Patch(fc="none", ec=C_CANEPHORA, hatch="////", label="Coffee mapped in OSM"),
    Patch(fc="none", ec="#9d9b94", label="Other cadastral property"),
    Line2D([], [], marker="o", ls="", color=C_FARM, mec="white", label="Farm in OSM (place=farm / 'Fazenda …')"),
    Line2D([], [], color=INK, lw=1.6, label="Municipal boundary"),
]
side.legend(handles=handles, loc="upper left", frameon=False, fontsize=9,
            title="Share of property under coffee", alignment="left")
top10 = coffee_farms.head(10)
txt = "\n".join(f"{r.coffee_rank:>2}. {str(r.label)[:28]:<28} {r.coffee_ha:>6,.0f} ha" for r in top10.itertuples())
if txt:
    side.text(0.0, 0.55, "Largest mapped coffee areas\n\n" + txt, transform=side.transAxes, family="monospace",
              fontsize=8, va="top", color=INK)
fig.text(0.02, 0.02, "Coffee & farms: OpenStreetMap (ODbL). Properties: SICAR/CAR. Basemap © OpenStreetMap contributors.",
         color=MUTED, fontsize=8)
fig.savefig(OUT_DIR / "04_coffee_fazendas.png", bbox_inches="tight")
plt.show()

In [ ]:
# Map C — close-up around the largest coffee fazenda and its neighbours
if len(coffee_web):
    focus = coffee_web.iloc[0]
    win = gpd.GeoSeries([focus.geometry.buffer(3500 / np.cos(np.radians(boundary.geometry.iloc[0].centroid.y)))], crs=WEB)
    fig, ax = plt.subplots(figsize=(10, 10))
    near = props_web[props_web.intersects(win.iloc[0])]
    near[~near["is_coffee"]].plot(ax=ax, facecolor="none", edgecolor=INK_2, linewidth=0.7)
    cw = coffee_web[coffee_web.intersects(win.iloc[0])]
    cw.plot(ax=ax, facecolor=BLUE_RAMP[2], alpha=0.35, edgecolor=BLUE_RAMP[4], linewidth=1.2)
    if len(coffee_poly_web):
        coffee_poly_web[coffee_poly_web.intersects(win.iloc[0])].plot(
            ax=ax, facecolor=C_OSM_COFFEE, alpha=0.45, edgecolor=C_OSM_COFFEE, linewidth=0.6)
    fw = farms_web[farms_web.intersects(win.iloc[0])]
    if len(fw):
        fw.plot(ax=ax, color=C_FARM, markersize=25, edgecolor="white", zorder=5)
        for _, r in fw[fw["name"].notna()].iterrows():
            ax.annotate(r["name"], (r.point.x, r.point.y), xytext=(5, 4), textcoords="offset points", fontsize=8,
                        color=INK, path_effects=[pe.withStroke(linewidth=2.5, foreground="white")])
    frame(ax, win, pad=0)
    basemap(ax, alpha=0.9)
    scalebar(ax, km=1)
    ax.set_title(f"Close-up: #{focus['coffee_rank']} {str(focus['label'])[:50]} "
                 f"— {focus['coffee_ha']:,.0f} ha of mapped coffee", loc="left")
    ax.legend(handles=[Patch(fc=BLUE_RAMP[2], alpha=0.35, ec=BLUE_RAMP[4], label="Coffee fazenda (CAR property)"),
                       Patch(fc=C_OSM_COFFEE, alpha=0.45, label="Coffee field in OSM"),
                       Patch(fc="none", ec=INK_2, label="Other property"),
                       Line2D([], [], marker="o", ls="", color=C_FARM, mec="white", label="OSM farm")],
              loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=9)
    fig.savefig(OUT_DIR / "05_closeup.png", bbox_inches="tight")
    plt.show()

### Interactive map (OpenStreetMap, with satellite imagery toggle)

In [ ]:
import folium
from folium.plugins import MarkerCluster, Fullscreen, MeasureControl


def to_wgs(gdf, tol_m=4):
    g = gdf.copy()
    g["geometry"] = g.geometry.simplify(tol_m) if g.crs == UTM else g.geometry
    return g.to_crs(4326)


center = boundary.geometry.iloc[0].representative_point()
m = folium.Map(location=[center.y, center.x], zoom_start=10, tiles=None, control_scale=True)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(m)
folium.TileLayer("CartoDB positron", name="OSM – light (CARTO)").add_to(m)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, Maxar, Earthstar Geographics", name="Satellite (Esri)").add_to(m)

folium.GeoJson(boundary[["name", "geometry"]], name="Municipal boundary",
               style_function=lambda f: {"color": INK, "weight": 3, "fill": False}).add_to(m)

other = to_wgs(props[~props["is_coffee"]][["property_id", "label", "size_class", "area_ha", "geometry"]])
other["size_class"] = other["size_class"].astype(str)
other["area_ha"] = other["area_ha"].round(1)
folium.GeoJson(other, name="Cadastral properties (CAR)", show=True,
               style_function=lambda f: {"color": "#6b6a66", "weight": 0.6, "fillOpacity": 0.03},
               tooltip=folium.GeoJsonTooltip(["label", "property_id", "size_class", "area_ha"],
                                             aliases=["Property", "CAR code", "Size", "Area (ha)"])).add_to(m)


def share_color(s):
    return share_colors[int(np.digitize(s, SHARE_BINS[1:-1]))]


cf = to_wgs(coffee_farms[["coffee_rank", "label", "property_id", "size_class", "area_ha", "coffee_ha",
                          "coffee_share", "geometry"]])
cf["size_class"] = cf["size_class"].astype(str)
cf[["area_ha", "coffee_ha"]] = cf[["area_ha", "coffee_ha"]].round(1)
cf["coffee_pct"] = (100 * cf["coffee_share"]).round(0)
folium.GeoJson(cf, name="Coffee fazendas",
               style_function=lambda f: {"color": "white", "weight": 0.8, "fillOpacity": 0.7,
                                         "fillColor": share_color(f["properties"]["coffee_share"])},
               highlight_function=lambda f: {"weight": 3, "color": INK},
               tooltip=folium.GeoJsonTooltip(
                   ["coffee_rank", "label", "property_id", "size_class", "area_ha", "coffee_ha", "coffee_pct"],
                   aliases=["Coffee rank", "Fazenda", "CAR code", "Size", "Property (ha)", "Coffee (ha)", "Coffee %"])
               ).add_to(m)

if len(coffee_poly):
    cp = to_wgs(coffee_poly[["osm_id", "name", "crop", "geometry"]])
    folium.GeoJson(cp, name="Coffee fields (OSM)",
                   style_function=lambda f: {"color": C_OSM_COFFEE, "weight": 1, "fillOpacity": 0.35},
                   tooltip=folium.GeoJsonTooltip(["osm_id", "name", "crop"])).add_to(m)

cluster = MarkerCluster(name="Farms in OSM").add_to(m)
for _, r in gpd.GeoDataFrame(osm_farms.drop(columns="geometry"), geometry="point", crs=UTM).to_crs(4326).iterrows():
    folium.CircleMarker([r.point.y, r.point.x], radius=5, color="white", weight=1, fill=True, fill_color=C_FARM,
                        fill_opacity=0.95, tooltip=r["name"] or r["osm_id"],
                        popup=f"<a href='https://www.openstreetmap.org/{r['osm_id']}' target='_blank'>{r['osm_id']}</a>"
                        ).add_to(cluster)

legend = f"""<div style="position:fixed;bottom:24px;left:12px;z-index:9999;background:white;padding:8px 10px;
 border:1px solid #ccc;border-radius:6px;font:12px system-ui,sans-serif;color:{INK}">
 <b>{NAME} ({UF}) — coffee fazendas</b><br>""" + "".join(
    f'<span style="display:inline-block;width:12px;height:12px;background:{c};margin-right:6px"></span>{l}<br>'
    for c, l in zip(share_colors, SHARE_LABELS)) + f"""
 <span style="display:inline-block;width:12px;height:12px;background:{C_OSM_COFFEE};opacity:.5;margin-right:6px"></span>Coffee field (OSM)<br>
 <span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:{C_FARM};margin-right:6px"></span>Farm (OSM)
 </div>"""
m.get_root().html.add_child(folium.Element(legend))
Fullscreen().add_to(m)
MeasureControl(primary_length_unit="kilometers", primary_area_unit="hectares").add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.fit_bounds([[BBOX[1], BBOX[0]], [BBOX[3], BBOX[2]]])
m.save(OUT_DIR / f"coffee_fazendas_{TARGET}.html")
m

## 7. Exports

In [ ]:
gpkg = OUT_DIR / f"coffee_fazendas_{TARGET}.gpkg"
boundary.to_file(gpkg, layer="municipality", driver="GPKG")
out_props = props.copy()
out_props["size_class"] = out_props["size_class"].astype(str)
out_props.to_crs(4326).to_file(gpkg, layer="cadastral_properties", driver="GPKG")
if len(coffee_poly):
    coffee_poly.to_crs(4326).to_file(gpkg, layer="osm_coffee_fields", driver="GPKG")
if len(osm_farms):
    gpd.GeoDataFrame(osm_farms.drop(columns="geometry"), geometry="point", crs=UTM).rename_geometry("geometry") \
        .to_crs(4326).to_file(gpkg, layer="osm_farms", driver="GPKG")
coffee_farms.drop(columns="geometry").assign(size_class=lambda d: d["size_class"].astype(str)) \
    .to_csv(OUT_DIR / f"coffee_fazendas_{TARGET}.csv", index=False)

summary = {
    "municipality": f"{NAME} ({UF})", "ibge_code": TARGET, "rank_in_brazil": int(target["rank"]),
    "mean_production_t": round(target["prod_t"]), "mean_bags_60kg": round(target["bags_60kg"]),
    "years": f"{RANK_YEARS[0]}–{RANK_YEARS[-1]}", "ibge_harvested_coffee_ha": round(target["harvested_ha"]),
    "cadastral_properties": len(props), "coffee_fazendas": len(coffee_farms),
    "osm_mapped_coffee_ha": round(OSM_COFFEE_HA),
    "osm_coverage_of_ibge_area_%": round(100 * OSM_COFFEE_HA / target["harvested_ha"], 1),
}
print(json.dumps(summary, indent=2, ensure_ascii=False))
print("\nFiles:", *sorted(p.name for p in OUT_DIR.iterdir()), sep="\n  ")